# Prefix programs: Colab runner

Runs the seminar grid on a Colab GPU. Every script is resumable: rerun a cell after a disconnect and it continues from `results/*.csv`.

**Before running:** set `GITHUB_TOKEN` (fine-grained PAT with read access to the repo) and `HF_TOKEN` (read) in Colab's Secrets panel, and accept the FLORES+ terms on the Hub once.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['RESULTS_DIR'] = '/content/drive/MyDrive/prefix-programs'
GH = userdata.get('GITHUB_TOKEN')
!git clone -q https://{GH}@github.com/sirotenkogeorgii/prefix-programs.git /content/prefix-programs 2>/dev/null || (cd /content/prefix-programs && git pull -q)
%cd /content/prefix-programs
!bash scripts/colab_bootstrap.sh

## 1. Pilot on `dev` (fixes operating points and the k grid before any devtest number)

Short runs on one cell per task to check that bits fall with k, pick the prompt-tuning learning rate, and time one run.

In [ ]:
!python scripts/run_sweep.py --model Qwen/Qwen2.5-1.5B --tasks translation,recall,continuation --langs ru \
    --ks 1,4,16 --seeds 0 --steps 300 --eval-every 50 --baselines --max-test 400 \
    --results results/pilot.csv --ckpt-dir checkpoints/pilot

In [ ]:
import pandas as pd
df = pd.read_csv('results/pilot.csv')
df[['cell','design','k','dev_bits_per_token','test_bits_per_token','test_bits_per_token_noncopy','best_step']]

## 2. Design A sweep, baselines and LoRA floors (WP3)

8 cells × k ∈ {1,2,4,8,16,32} × 2 seeds, plus `lang` and `task` conditioning at one seed.

In [ ]:
!python scripts/run_sweep.py --model Qwen/Qwen2.5-1.5B --tasks translation,recall,continuation --langs ru,de,en \
    --ks 1,2,4,8,16,32 --seeds 0,1 --conds none --baselines --floors --results results/sweep.csv

In [ ]:
# Conditioning: prefixes on top of the language label (lang) and the task instruction (task), one seed.
# 'both' is the natural-language baseline only (no prefix), as preregistered.
!python scripts/run_sweep.py --model Qwen/Qwen2.5-1.5B --tasks translation,recall,continuation --langs ru,de,en \
    --ks 1,2,4,8,16,32 --seeds 0 --conds lang,task --baselines --floors --results results/sweep.csv
!python scripts/run_sweep.py --model Qwen/Qwen2.5-1.5B --tasks translation,recall,continuation --langs ru,de,en \
    --ks "" --seeds 0 --conds both --baselines --results results/sweep.csv


## 3. Variational rates (WP4a): flat at k* ± neighbours, then hierarchical

Fill `--ks` from the k* table of step 2 (see `prefixlab.analysis.curves.k_star`).

In [ ]:
!python scripts/run_sweep.py --design flat --betas 0.3,1,3 --model Qwen/Qwen2.5-1.5B \
    --tasks translation,recall,continuation --langs ru,de,en --ks 2,4,8 --seeds 0 --results results/flat.csv

In [ ]:
!python scripts/run_hier.py --group task --model Qwen/Qwen2.5-1.5B --tasks translation,recall,continuation --langs ru,de,en \
    --ks 2,4,8 --betas 0.3,1,3 --taus 1 --results results/hier_task.csv
!python scripts/run_hier.py --group lang --model Qwen/Qwen2.5-1.5B --tasks translation,recall,continuation --langs ru,de,en \
    --ks 2,4,8 --betas 0.3,1,3 --taus 1 --results results/hier_lang.csv

## 4. Composition, distances, regime 2, mechanism (WP4b/WP5)

All at k = 4 (the middle of the flat / hierarchical grid), seed 0, no conditioning. Needs the Design A rows of step 2 and, for the
hierarchical probes of 4d, the step 3 tables. Roughly 0.5 h + 2 h + 1 h + 0.7 h on an A100.

In [ ]:
# 4a. Composition: tied task parts + language parts, total slots 2/4/8; one run trains on every cell, one holds out two cells.
!python scripts/run_compose.py --model Qwen/Qwen2.5-1.5B --tasks translation,recall,continuation --langs ru,de,en \
    --k-pairs 1+1,2+2,4+4 --seeds 0 --conds none --holdout-sets none,recall-de+continuation-ru --results results/compose.csv


In [ ]:
# 4b. Distance matrix: warm-start from every cell's k=4 prefix (and from scratch) toward every cell, 512-example budget, 50 steps per block.
!python scripts/run_distance.py --model Qwen/Qwen2.5-1.5B --tasks translation,recall,continuation --langs ru,de,en \
    --k 4 --seed 0 --cond none --sweep-results results/sweep.csv --budget 512 --first-block 16 --steps 50 --max-test 512 \
    --results results/distance.csv


In [ ]:
# 4c. Regime 2: 100 test examples x 4 cells, m in {1,2,4,8}, without and with the shared k=4 prefix.
!python scripts/run_regime2.py --model Qwen/Qwen2.5-1.5B --cells translation-ru,translation-de,recall-ru,continuation-ru \
    --ms 1,2,4,8 --n-examples 100 --k 0 --results results/regime2.csv
!python scripts/run_regime2.py --model Qwen/Qwen2.5-1.5B --cells translation-ru,translation-de,recall-ru,continuation-ru \
    --ms 1,2,4,8 --n-examples 100 --k 4 --sweep-results results/sweep.csv --results results/regime2.csv


In [ ]:
# 4d. Mechanism: per-block window patching at k=4; probes = Design A prefix, hierarchical centre alone, deviation on top of the centre.
!python scripts/run_mechanism.py --model Qwen/Qwen2.5-1.5B --tasks translation,recall,continuation --langs ru,de,en \
    --k 4 --seed 0 --cond none --sweep-results results/sweep.csv --hier-results results/hier_task.csv,results/hier_lang.csv \
    --hier-beta 1 --hier-tau 1 --max-examples 256 --results results/mechanism.csv


## Sync results to GitHub
Run any time; safe to repeat. Pull on your machine afterwards to analyse.

In [ ]:
# Sync result tables (CSVs + per-example JSONs, no checkpoints) into the repo so they can be analysed locally.
from google.colab import userdata
tok = userdata.get('GITHUB_TOKEN')
!mkdir -p results_sync && cp results/*.csv results_sync/ && cp -r results/per_example results_sync/
!git config user.email 'colab@users.noreply.github.com' && git config user.name 'Colab runner'
!git add results_sync && (git commit -q -m 'Sync results from Colab' || echo 'nothing new')
!git push -q https://{tok}@github.com/sirotenkogeorgii/prefix-programs.git HEAD:prefixlab
